# LangChain Tool 实践经验总结
## 1. 清晰的描述（工具文档规范）
### ✅ 规范示例
```python
# ✅ 好
@tool(parse_docstring=True)
def search_flights(origin: str, destination: str, date: str) -> str:
    """
    搜索航班信息

    Args:
        origin: 出发城市，如"北京"
        destination: 目的地城市，如"上海"
        date: 出发日期，格式 YYYY-MM-DD

    Returns:
        可用航班的 JSON 列表
    """
```

## 2. 功能单一原则
### ❌ 反面：一个工具做多件事
```python
# ❌ 不好：一个工具做太多事
@tool
def do_everything(action: str, data: str) -> str:
    """做各种事情"""
    if action == "weather": ...
    elif action == "calculate": ...
    elif action == "search": ...
```

### ✅ 正面：每个工具只做一件事
```python
# ✅ 好：每个工具做一件事
@tool
def get_weather(city: str) -> str:
    """获取天气"""
    ...

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """计算"""
    ...
```

## 3. 工具失败三层防护方案
### 第1层：工具内部异常捕获
```python
@tool
def divide(a: float, b: float) -> str:
    """
    除法计算

    Args:
        a: 被除数
        b: 除数
    """
    try:
        if b == 0:
            return "错误：除数不能为零"
        result = a / b
        return f"{a} / {b} = {result}"
    except Exception as e:
        return f"计算错误：{e}"
```

### 第2层：Agent 提示词重试引导
```python
agent = create_agent(
    model=model,
    tools=[...],
    prompt="如果工具失败，尝试使用其他方法解决问题。"
)
```

### 第3层：调用级重试（tenacity @retry）
```python
from tenacity import retry, stop_after_attempt

# 1. 配置重试规则：失败最多尝试3次（1次初始+2次重试）
@retry(stop=stop_after_attempt(3))
def call_agent(question):
    # 2. 核心业务逻辑：调用 LangChain 的 Agent
    return agent.invoke({"messages": [{"role": "user", "content": question}]})
```
#### 重试工作流程
1. 调用 `call_agent("你好")`
2. 进入函数执行 `agent.invoke(...)`
3. 执行成功：直接返回结果，`@retry` 无操作
4. 执行报错：`@retry` 拦截异常，自动重新执行函数
5. 连续3次失败：抛出原始异常，程序终止

## 4. 返回值统一使用字符串 str
### ✅ 推荐写法（返回JSON字符串）
```python
# ✅ 好：返回字符串
import json
@tool
def get_user_info(user_id: str) -> str:
    """获取用户信息"""
    user = {"id": user_id, "name": "张三"}
    return json.dumps(user, ensure_ascii=False)  # 转成 JSON 字符串
```

### ❌ 不推荐（直接返回字典）
```python
# ❌ 不好：返回字典（部分场景存在格式问题）
@tool
def get_user_info(user_id: str) -> dict:
    """获取用户信息"""
    return {"id": user_id, "name": "张三"}
```

### 核心说明
1. LLM 底层只识别文本，字典会被强制转义 Unicode，中文变成乱码 `\u5f20\u4e09`，干扰模型理解；
2. `json.dumps(ensure_ascii=False)` 保留中文原始文本，输入给大模型更稳定；
3. `json.dumps` 作用：将 Python 对象（字典/列表）序列化为标准JSON字符串；
4. `ensure_ascii=False`：关闭中文转 Unicode，中文、表情正常显示。

## 5. 同步工具 vs 异步工具 选型
### 适用场景
- **同步工具**：简单计算、CPU密集型任务
- **异步工具**：IO密集型（API调用、数据库、文件读写）

### 代码示例
```python
# 同步工具
@tool
def sync_tool(x: str) -> str:
    return sync_process(x)

# 异步工具
@tool
async def async_tool(x: str) -> str:
    return await async_process(x)
```

## 核心总结要点
1. 工具文档必须清晰：写明入参含义、返回格式，用 `parse_docstring=True` 自动生成Schema；
2. 单一职责：一个工具只实现一类能力，不要用分支判断承载多业务；
3. 失败三层兜底：工具内try-except → Agent提示引导重试 → tenacity调用层重试；
4. 返回统一JSON字符串，避免原生dict造成中文转义乱码；
5. IO请求优先写异步工具，纯计算使用同步工具。